<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "780"
iso_country = "Trinidad and Tobago"
year = 2025

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Trinidad and Tobago': 

Number of trade records present in dataset: 26201
Number of partner countries present in dataset: 165
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$5073.1m
Total weight within the dataset: 6395.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,780,Trinidad and Tobago,862,Venezuela,Import,Sea,392390,392390 | Plastics; articles f,3923,3923 | Plastic articles for,39,39 | Plastics and article,3649.0,Other articles for the conveyance or packing o...,"Petroleum, Chemical and Non-Metallic Mineral P...",33.0,"CRP - Chemical, rubber, plastic products",2.223,NaN,2.223,kg,1.0,1.0,0.586872,631.0,392.085101,0.230104,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",16.17,1,0.31,2.22,1.91,1. Primary Int. Inv. with Vess Type,0.001105,0.001103,0.001103,0.001103,0.001104,0.001104,0.001157,0.002071,0.001276,0.001183,0.001187,0.001217,0.001132,0.002018,0.001341,0.001229,0.001201,0.001256,0.001111,0.002202,0.001666,0.001456,0.001347,0.00131,0.001090,0.002340,0.001917,0.001795,0.001439,0.001405,0.001069,0.002374,0.002264,0.002286,0.001560,0.001476,0.001048,0.002326,0.002228,0.002428,0.001801,0.001503,0.001028,0.002073,0.002156,0.002258,0.001689,0.001521,0.001008,0.001898,0.001931,0.002378,0.001965,0.001532,3649
1,1,780,Trinidad and Tobago,862,Venezuela,Import,Sea,482390,"482390 | Paper pulp, paper, p",4823,"4823 | Paper, paperboard, c",48,48 | Paper and paperboard,32199.0,"Other paper, paperboard, cellulose wadding and...",Wood and Paper,31.0,"PPP - Paper products, publishing",2.075,NaN,2.075,kg,1.0,1.0,0.747830,631.0,392.085101,0.293213,['all'],['all'],"1 - Reporter, Partner and HS4 code",2.82,1,0.06,2.08,2.02,3. Primary Int. Inv. without Vess Type,0.001130,0.001129,0.001129,0.001129,0.001129,0.001133,0.001149,0.001873,0.001214,0.001155,0.001157,0.001190,0.001127,0.001810,0.001263,0.001192,0.001163,0.001213,0.001112,0.001993,0.001497,0.001371,0.001280,0.00126,0.001097,0.002203,0.001758,0.001652,0.001345,0.001341,0.001081,0.002192,0.002070,0.002139,0.001436,0.001403,0.001064,0.002104,0.002036,0.002182,0.001712,0.001425,0.001048,0.001932,0.001982,0.002115,0.001683,0.001458,0.001029,0.001780,0.001824,0.002101,0.001741,0.001466,32199


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1299.7m (25.6%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
04 | Dairy produce; birds  152204463.1   54302246.5     11.7      4.6
21 | Miscellaneous edible  142105795.8   34020895.7     10.9      2.9
02 | Meat and edible meat  123915045.7   40445938.6      9.5      3.4
15 | Animal, vegetable or   95837055.4   83166924.2      7.4      7.1
20 | Preparations of vege   93725301.7   67103805.5      7.2      5.7
10 | Cereals                86539109.0  273382678.4      6.7     23.2
22 | Beverages, spirits a   80622538.3   85514122.1      6.2      7.3
23 | Food industries, res   79162885.1   90306521.0      6.1      7.7
19 | Preparations of cere   77710807.8   28019819.5      6.0      2.4
17 | Sugars and sugar con   72001045.5  149986182.2      5.5     12.7
07 | Vegetables and certa   68213699.6  117822566.9      5

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [14]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1169.1m (23.0%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  608880171.0  438560194.0     12.0      6.9
Agriculture       551605599.4  528875098.7     10.9      8.3
Fishing             8592909.6    1250491.8      0.2      0.0



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1083.4m (21.4%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     492289771.3  269211638.3      9.7      4.2
MIL - Dairy products           122337848.8   36716779.8      2.4      0.6
VOL - Vegetable oils and fats  109470415.5  123616349.6      2.2      1.9
V_F - Vegetables, fruit, nuts   97510402.5  132535728.7      1.9      2.1
OMT - Meat products n.e.c.      83911832.1   31459192.8      1.7      0.5
CMT - Bovine meat prods         66565566.4   14817305.0      1.3      0.2
SGR - Sugar                     43812958.5  114181713.5      0.9      1.8
GRO - Cereal grains n.e.c.      24232233.4   99792341.5      0.5      1.6
PCR - Processed rice            14665568.4   25474092.2      0.3      0.4
OCR - Crops n.e.c.              14092428.8    4400986.1      0.3      0.1
PDR - Paddy rice                12583213.7   

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 25.6% 
GTAP 21.4% 
EORA 23.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21398/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,780,Trinidad and Tobago,862,Venezuela,Import,Sea,392390,392390 | Plastics; articles f,3923,3923 | Plastic articles for,39,39 | Plastics and article,3649.0,Other articles for the conveyance or packing o...,"Petroleum, Chemical and Non-Metallic Mineral P...",33.0,"CRP - Chemical, rubber, plastic products",2.223,NaN,2.223,kg,1.0,1.0,0.586872,631.0,392.085101,0.230104,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",16.17,1,0.31,2.22,1.91,1. Primary Int. Inv. with Vess Type,0.001105,0.001103,0.001103,0.001103,0.001104,0.001104,0.001157,0.002071,0.001276,0.001183,0.001187,0.001217,0.001132,0.002018,0.001341,0.001229,0.001201,0.001256,0.001111,0.002202,0.001666,0.001456,0.001347,0.00131,0.001090,0.002340,0.001917,0.001795,0.001439,0.001405,0.001069,0.002374,0.002264,0.002286,0.001560,0.001476,0.001048,0.002326,0.002228,0.002428,0.001801,0.001503,0.001028,0.002073,0.002156,0.002258,0.001689,0.001521,0.001008,0.001898,0.001931,0.002378,0.001965,0.001532,3649,NaN
1,1,780,Trinidad and Tobago,862,Venezuela,Import,Sea,482390,"482390 | Paper pulp, paper, p",4823,"4823 | Paper, paperboard, c",48,48 | Paper and paperboard,32199.0,"Other paper, paperboard, cellulose wadding and...",Wood and Paper,31.0,"PPP - Paper products, publishing",2.075,NaN,2.075,kg,1.0,1.0,0.747830,631.0,392.085101,0.293213,['all'],['all'],"1 - Reporter, Partner and HS4 code",2.82,1,0.06,2.08,2.02,3. Primary Int. Inv. without Vess Type,0.001130,0.001129,0.001129,0.001129,0.001129,0.001133,0.001149,0.001873,0.001214,0.001155,0.001157,0.001190,0.001127,0.001810,0.001263,0.001192,0.001163,0.001213,0.001112,0.001993,0.001497,0.001371,0.001280,0.00126,0.001097,0.002203,0.001758,0.001652,0.001345,0.001341,0.001081,0.002192,0.002070,0.002139,0.001436,0.001403,0.001064,0.002104,0.002036,0.002182,0.001712,0.001425,0.001048,0.001932,0.001982,0.002115,0.001683,0.001458,0.001029,0.001780,0.001824,0.002101,0.001741,0.001466,32199,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $826.0m of $5073.1m (16.3%)
Volume: 844.4 of 6395.2 million kilograms (13.2%)
Value-to-Weight: 0.98 from 0.79 US$ per kg (+23.3%)
Average distance to market (volume-weighted): 3010.9 from 3944.3 nm (+-23.7%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    610  278906447.2  166900441.6      5.5      2.6
1                Dairy    130  123609780.6   39240621.1      2.4      0.6
2           Vegetables    353   91369201.3  137055682.9      1.8      2.1
3        Fats and Oils    142   78274675.2   62467993.8      1.5      1.0
4        Meat and Prep     43   74782398.1   28283133.9      1.5      0.4
5      Sugar and Honey    139   72153895.1  150030004.2      1.4      2.3
6  Unprocessed Cereals     51   59290326.9  226051543.9      1.2      3.5
7               Fruits    166   47606034.6   34386162.3      0.9      0.5



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [16]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,97218384.0,172646846.6,106776838.8,99304979.7,99584188.1,102148842.7,90003764.6,198219375.7,188876785.6,191114408.2,130060822.2,123613466.0,84910917.9,158690302.5,161568623.7,197794604.0,163436191.0,128399660.5
1,Fats and Oils,127454005.3,209327676.0,136333976.4,127785690.3,128147327.7,132576510.8,119466976.3,236162074.7,228247993.0,236093569.4,164101707.9,155678890.7,113881997.0,195462810.6,199704439.4,234210045.0,194051795.5,160573434.4
2,Fruits,66055410.0,118221929.7,72841979.6,67576008.4,67771987.8,69508198.8,61048470.6,135536735.9,129233260.7,130550617.3,89072508.2,84286269.4,57553224.6,108361909.1,110260517.9,135737011.1,112176704.4,87454927.9
3,Meat and Prep,47009689.5,84165213.6,51847577.3,48095229.7,48234837.2,49469786.9,43443389.7,96489250.3,92002824.5,92932909.3,63407300.9,59994547.9,40954370.0,77135748.8,78485747.1,96642790.0,79868804.7,62249319.7
4,Processed Cereals,232947829.6,408996012.2,254820178.9,237368274.4,238047142.9,244401989.9,216065124.6,471154009.5,448805407.1,455389297.7,311041733.4,294341046.6,204174898.6,378163690.3,385380360.7,468230240.1,387716002.7,305349611.4
5,Sugar and Honey,108165006.1,182821462.0,116755425.1,109277675.9,109604370.2,112927878.4,100943653.2,211351327.8,201951591.4,206823243.8,142990805.4,133919431.3,95881649.9,171770328.1,175526042.9,208227726.7,173272016.3,138366658.1
6,Unprocessed Cereals,231184834.9,405758205.8,253072715.9,235505973.3,236205487.7,242620184.0,214379436.7,467753504.4,445982764.5,452502567.0,310389122.1,291886818.1,202647507.3,375722478.9,382962426.2,465238073.1,385762845.1,302322597.5
7,Vegetables,250435023.0,446143114.0,275564572.6,255966020.0,256698916.1,263313958.4,231672935.9,512021125.8,487970232.0,493485657.8,336419536.5,318842439.1,218518135.7,409659790.0,416978403.7,511680813.4,422931248.5,330944855.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [15]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.61,0.08,0.02,0.02,0.04,0.88,0.80,0.82,0.32,0.27,0.60,0.62,0.91,0.64,0.35
1,Fats and Oils,1.05,0.11,0.00,0.01,0.07,1.49,1.39,1.49,0.57,0.46,1.04,1.10,1.54,1.02,0.60
2,Fruits,1.10,0.14,0.03,0.04,0.07,1.56,1.43,1.46,0.59,0.49,1.07,1.11,1.64,1.15,0.63
3,Meat and Prep,0.50,0.06,0.01,0.02,0.03,0.71,0.65,0.66,0.27,0.22,0.48,0.50,0.74,0.52,0.28
4,Processed Cereals,0.63,0.08,0.02,0.02,0.04,0.91,0.83,0.86,0.34,0.28,0.62,0.65,0.95,0.66,0.36
5,Sugar and Honey,1.03,0.12,0.02,0.02,0.07,1.53,1.40,1.47,0.58,0.46,1.05,1.10,1.56,1.07,0.59
6,Unprocessed Cereals,2.94,0.37,0.07,0.08,0.19,4.27,3.91,4.02,1.62,1.31,2.92,3.04,4.43,3.09,1.68
7,Vegetables,2.14,0.28,0.06,0.07,0.14,3.07,2.81,2.87,1.15,0.95,2.09,2.17,3.21,2.24,1.23
